# 👄 Lectura de labios multilingüe (VSR) — demo en Colab

**Qué hace:** recibe un vídeo de una persona hablando y **transcribe lo que dice SIN usar el audio**,
solo mirando el movimiento de los labios (*Visual Speech Recognition*). Con selector de idioma.

**Modelo:** [`mpc001/Visual_Speech_Recognition_for_Multiple_Languages`](https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages)
— pesos preentrenados públicos. Solo **inferencia**, no se entrena nada.

---

### 📱 Cómo usarlo desde el móvil (Samsung), en orden
1. Abre este notebook en **Google Colab**.
2. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** → Guardar.
3. Ejecuta las celdas **de arriba abajo, una a una**, esperando a que cada una termine.
4. Trabaja **por fases**: cada fase tiene una celda de **✅ VERIFICACIÓN**. No sigas si una falla.
5. En la **Fase 4** aparecerá un **enlace público de Gradio** (`https://....gradio.live`). Ábrelo en el
   navegador del móvil y sube tu vídeo.

> 🔬 **Qué está verificado y qué no.** El preprocesado completo (MediaPipe → recorte de labios 96×96)
> se **ejecutó de verdad en CPU** durante el desarrollo, con vídeo real, y los parches de compatibilidad
> de abajo salen de errores reales cazados en esa ejecución (no de suposiciones). Lo que NO se pudo
> ejecutar en desarrollo es la **inferencia con los pesos** (Google Drive estaba bloqueado en ese
> entorno): eso se verifica aquí en Colab, en las Fases 1 y 3.


## 🗺️ Fase 0 — Plan del notebook (léelo antes de ejecutar)

- **Fase 1** — Entorno + modelo español: GPU, dependencias, **parches de compatibilidad**, clonado,
  pesos de español con `gdown`. *Verifica:* versiones + modelo español cargado en GPU.
- **Fase 2** — Preprocesado: MediaPipe (NO dlib) → recorte de boca. *Verifica:* 5 fotogramas del recorte.
  **Trae vídeos de prueba incluidos** (extraídos de los GIFs de demo del propio repo).
- **Fase 3** — Inferencia español + inglés. *Verifica:* texto obtenido vs real y % de acierto.
- **Fase 4** — Interfaz **Gradio** con `share=True`.
- **Fase 5** — Francés, portugués y mandarín bajo demanda.

**Riesgos de compatibilidad — ya no son teoría, están CAZADOS y parcheados (ver celda 1.5):**
1. `torchvision` moderno **eliminó `read_video`**, que el repo usa para leer vídeos → reponemos la
   función con PyAV. (Error real reproducido en desarrollo con torchvision 0.28.)
2. MediaPipe moderno **eliminó la API `solutions`** que usa el repo (y las versiones antiguas no
   existen para el Python del Colab actual o chocan con su protobuf) → usamos MediaPipe moderno y
   la celda 1.5 **inyecta un detector equivalente con la API nueva "tasks"** (verificado: 597/611
   fotogramas con cara en el vídeo de prueba, recorte idéntico).
3. PyTorch ≥2.6 cambió `torch.load` a `weights_only=True` por defecto → los `torch.load` del repo
   podrían fallar al cargar el modelo de lenguaje → parche que restaura el comportamiento antiguo.
4. Google Drive a veces da "límite de descarga superado" → reintenta más tarde o usa el espejo
   Zenodo (doi 10.5281/zenodo.7065080).

> ❗ **El italiano NO tiene pesos públicos en este repo** (solo inglés, español, mandarín, francés y
> portugués). Queda fuera del selector en vez de inventarlo.

**Combinación verificada en desarrollo (CPU):** Python 3.11 · torch 2.13 · torchvision 0.28 ·
numpy 2.4 · mediapipe 0.10.9 · av 18 → el preprocesado funcionó con los parches de la celda 1.5.


## 🔧 Fase 1 — Entorno + modelo (español)


In [ ]:
# 1.1 — ¿Tenemos GPU T4? (si falla: Entorno de ejecución → Cambiar tipo → GPU)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1.2 — Instalar dependencias.
# NO tocamos el torch de Colab (reinstalarlo rompe la GPU). Claves:
# - mediapipe: el MODERNO (el antiguo no existe para el Python de Colab actual y el de
#   en medio choca con el protobuf de Colab). El repo usa una API antigua de mediapipe,
#   pero la celda de parches (1.5) inyecta un detector equivalente con la API nueva.
# - av: para leer vídeo (y para nuestro parche de read_video).
# - sentencepiece: tokenizador del modelo de lenguaje. gdown: descarga de Google Drive.
%pip install -q --upgrade mediapipe
%pip install -q hydra-core av scikit-image sentencepiece gdown six
print("✅ Instalación terminada")

In [ ]:
# 1.3 — ¿Qué mediapipe tenemos? (solo informativo; los parches de 1.5 cubren ambos casos)
import mediapipe as mp
modo = "API clásica 'solutions'" if hasattr(mp, "solutions") else "API moderna 'tasks' (usaremos nuestro detector adaptado)"
print("✅ mediapipe", mp.__version__, "→", modo)

In [ ]:
# 1.4 — Clonar el repositorio oficial del modelo.
import os
if not os.path.isdir("Visual_Speech_Recognition_for_Multiple_Languages"):
    !git clone --depth 1 https://github.com/mpc001/Visual_Speech_Recognition_for_Multiple_Languages
%cd Visual_Speech_Recognition_for_Multiple_Languages
!ls

In [ ]:
# 1.5 — 🩹 PARCHES DE COMPATIBILIDAD (salen de errores REALES, no de teoría).
# Ejecuta esta celda SIEMPRE antes de importar nada de "pipelines".
import numpy as np, torch, torchvision.io

# (a) torchvision moderno eliminó read_video (el repo lo usa para leer los vídeos).
#     Lo reponemos con PyAV: mismos fotogramas RGB, misma forma de tensor.
if not hasattr(torchvision.io, "read_video"):
    import av as _av
    def _read_video(filename, pts_unit="sec", **kw):
        contenedor = _av.open(filename)
        fotogramas = [f.to_ndarray(format="rgb24") for f in contenedor.decode(video=0)]
        return torch.from_numpy(np.stack(fotogramas)), None, {}
    torchvision.io.read_video = _read_video
    print("🩹 (a) torchvision.io.read_video repuesto con PyAV")

# (b) PyTorch >= 2.6 cambió torch.load a weights_only=True por defecto; los checkpoints
#     del repo (sobre todo el modelo de lenguaje) pueden necesitar el modo antiguo.
_torch_load_original = torch.load
def _torch_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _torch_load_original(*args, **kwargs)
torch.load = _torch_load
print("🩹 (b) torch.load con weights_only=False por defecto")

# (c) NumPy >= 1.24 eliminó los alias np.int / np.float / np.bool que usa espnet en
#     algunos caminos (los modelos transformer no pasan por ahí, pero el parche es gratis).
for _alias, _tipo in [("int", int), ("float", float), ("bool", bool), ("object", object)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _tipo)
print("🩹 (c) alias clásicos de numpy restaurados")

# (d) mediapipe moderno eliminó la API "solutions" que usa el detector del repo.
#     Si es el caso, inyectamos un detector equivalente con la API nueva "tasks"
#     (misma salida: por fotograma, None o array 4x2 [ojo dcho, ojo izq, nariz, boca] en píxeles).
import mediapipe as mp
if not hasattr(mp, "solutions"):
    import os as _os
    from mediapipe.tasks import python as mp_tasks
    from mediapipe.tasks.python import vision as mp_vision
    _MODELO_CARAS = "/content/blaze_face_short_range.tflite"
    if not _os.path.exists(_MODELO_CARAS):
        import urllib.request as _ur
        for _u in ("https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
                   "https://raw.githubusercontent.com/tejex/faceAnonymizer/main/blaze_face_short_range.tflite"):
            try:
                _ur.urlretrieve(_u, _MODELO_CARAS); break
            except Exception:
                continue
    class _DetectorModerno:
        def __init__(self):
            self._d = mp_vision.FaceDetector.create_from_options(
                mp_vision.FaceDetectorOptions(
                    base_options=mp_tasks.BaseOptions(model_asset_path=_MODELO_CARAS),
                    min_detection_confidence=0.5))
        def __call__(self, filename):
            video = torchvision.io.read_video(filename, pts_unit="sec")[0].numpy()
            out = []
            for frame in video:
                img = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame))
                res = self._d.detect(img)
                if not res.detections:
                    out.append(None); continue
                ih, iw = frame.shape[:2]
                mejor = max(res.detections, key=lambda d: d.bounding_box.width + d.bounding_box.height)
                kp = mejor.keypoints
                out.append(np.array([[int(kp[i].x * iw), int(kp[i].y * ih)] for i in range(4)]))
            return out
    import pipelines.detectors.mediapipe.detector as _det_mod
    _det_mod.LandmarksDetector = _DetectorModerno
    print("🩹 (d) detector de caras adaptado a la API moderna de mediapipe")
print("✅ Parches aplicados")

In [ ]:
# 1.6 — Catálogo de idiomas: config real + enlaces reales del "model zoo" del repo.
# Cada idioma necesita 2 descargas: MODELO visual y MODELO DE LENGUAJE.
# Con MediaPipe calculamos los landmarks al vuelo: NO hace falta el zip de landmarks.
IDIOMAS = {
    "es": {"nombre": "Español",   "config": "configs/CMUMOSEAS_V_ES_WER44.5.ini",
           "modelo": "https://bit.ly/34MjWBW", "lm": "https://bit.ly/3rppyJN", "wer": 44.5},
    "en": {"nombre": "Inglés",    "config": "configs/LRS3_V_WER19.1.ini",
           "modelo": "https://bit.ly/3Bp4gjV", "lm": "https://bit.ly/3qzWKit", "wer": 19.1},
    "fr": {"nombre": "Francés",   "config": "configs/CMUMOSEAS_V_FR_WER58.6.ini",
           "modelo": "https://bit.ly/3Ik6owb", "lm": "https://bit.ly/3LDChSn", "wer": 58.6},
    "pt": {"nombre": "Portugués", "config": "configs/CMUMOSEAS_V_PT_WER51.4.ini",
           "modelo": "https://bit.ly/3HjXCgo", "lm": "https://bit.ly/3gPvneF", "wer": 51.4},
    "zh": {"nombre": "Mandarín",  "config": "configs/CMLR_V_WER8.0.ini",
           "modelo": "https://bit.ly/3fR8RkU", "lm": "https://bit.ly/3fPxXAJ", "wer": 8.0},
    # Italiano: SIN pesos públicos en este repo -> no disponible (no lo inventamos).
}
print("Idiomas disponibles:", [v["nombre"] for v in IDIOMAS.values()])

In [ ]:
# 1.7 — Descarga de pesos: bit.ly -> Google Drive -> gdown, y descomprimir en su sitio.
import os, zipfile, configparser, urllib.request, gdown

def _resolver_bitly(url):
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req) as r:
        return r.url

def _descargar_y_descomprimir(url, etiqueta):
    real = _resolver_bitly(url)
    print(f"  {etiqueta}: {real}")
    destino = f"/tmp/{etiqueta}.zip"
    gdown.download(url=real, output=destino, quiet=False, fuzzy=True)
    if zipfile.is_zipfile(destino):
        with zipfile.ZipFile(destino) as z:
            z.extractall(".")   # crea benchmarks/<DATASET>/... como esperan los .ini
        print(f"  ✔ {etiqueta} descomprimido")
    else:
        print(f"  ⚠ {etiqueta} NO es un zip (¿cuota de Drive superada?). Prueba más tarde o Zenodo.")

def preparar_idioma(cod):
    """Descarga modelo + LM de un idioma SOLO si faltan. Devuelve la ruta del config."""
    info = IDIOMAS[cod]
    cfg = configparser.ConfigParser(); cfg.read(info["config"])
    rutas = [cfg["model"]["model_path"], cfg["model"]["rnnlm"]]
    if all(os.path.exists(r) for r in rutas):
        print(f"✅ {info['nombre']}: pesos ya presentes."); return info["config"]
    print(f"⬇️  Descargando pesos de {info['nombre']}...")
    _descargar_y_descomprimir(info["modelo"], f"{cod}_modelo")
    _descargar_y_descomprimir(info["lm"],     f"{cod}_lm")
    for r in rutas:
        print(("  ✔" if os.path.exists(r) else "  ❌ FALTA"), r)
    if not all(os.path.exists(r) for r in rutas):
        print("  ⚠ Si falta algo: mira con !find benchmarks -name '*.pth' dónde quedó y muévelo.")
    return info["config"]

print("Ayudantes de descarga listos.")

In [ ]:
# 1.8 — Descargar SOLO el español (empezamos simple).
config_es = preparar_idioma("es")

In [ ]:
# ✅ VERIFICACIÓN Fase 1 — versiones + el modelo español carga en GPU sin errores.
import torch, sys, mediapipe
print("Python   :", sys.version.split()[0])
print("PyTorch  :", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")
print("MediaPipe:", mediapipe.__version__)

from pipelines.pipeline import InferencePipeline
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
pipeline_es = InferencePipeline(config_es, detector="mediapipe", face_track=True, device=DEVICE)
print("\n✅ Modelo ESPAÑOL cargado en", DEVICE, "— Fase 1 superada.")

## 🎞️ Fase 2 — Preprocesado del vídeo (recorte de labios / ROI)

El modelo no mira el vídeo entero: mira **solo un recuadro de la boca de 96×96 en escala de grises**.
Flujo: fotogramas → MediaPipe detecta la cara → landmarks → recorte normalizado.

**No necesitas grabar nada para verificar esta fase:** el propio repo trae dos GIFs de demo con caras
reales hablando (uno en **inglés** y otro en **francés**, con el texto real subtitulado). Los convertimos
a MP4 y los usamos de vídeos de prueba. *(Este truco se probó en desarrollo: 600/611 fotogramas con cara
detectada y recorte correcto.)*


In [ ]:
# 2.1 — Crear los vídeos de prueba desde los GIFs de demo del propio repo (PyAV, sin ffmpeg).
import av, os

def gif_a_mp4(src, dst, max_fotogramas=None):
    entrada = av.open(src)
    fotogramas = [f.to_ndarray(format="rgb24") for f in entrada.decode(video=0)]
    if max_fotogramas:
        fotogramas = fotogramas[:max_fotogramas]
    salida = av.open(dst, "w")
    try:
        stream = salida.add_stream("h264", rate=25)
    except Exception:
        stream = salida.add_stream("mpeg4", rate=25)
    stream.width, stream.height = fotogramas[0].shape[1], fotogramas[0].shape[0]
    stream.pix_fmt = "yuv420p"
    for fr in fotogramas:
        for pkt in stream.encode(av.VideoFrame.from_ndarray(fr, format="rgb24")):
            salida.mux(pkt)
    for pkt in stream.encode():
        salida.mux(pkt)
    salida.close()
    print(f"  ✔ {dst}: {len(fotogramas)} fotogramas, {os.path.getsize(dst)//1024} KB")

# Recortamos a los primeros ~6 s (150 fotogramas): el primer plano de cada GIF, cuyo
# subtítulo conocemos. (Los GIFs son montajes de varios clips; mejor un solo hablante.)
gif_a_mp4("doc/vsr_1.gif", "/content/test_en.mp4", max_fotogramas=150)   # inglés
gif_a_mp4("doc/vsr_2.gif", "/content/test_fr.mp4", max_fotogramas=150)   # francés
print("✅ Vídeos de prueba creados en /content/")

In [ ]:
# 2.2 — Vídeo para verificar el recorte. Por defecto el de inglés del repo.
# Si quieres usar el tuyo: súbelo (panel 📁 → subir) y cambia la ruta.
VIDEO_PRUEBA = "/content/test_en.mp4"
import os
assert os.path.exists(VIDEO_PRUEBA), "No encuentro el vídeo."
print("Usando:", VIDEO_PRUEBA)

In [ ]:
# 2.3 — Recorte de labios REAL: landmarks primero, y cargador con transform=False
# (así vemos la imagen cruda; con la transformación del modelo saldrían números, no imagen).
import numpy as np, cv2
from pipelines.detectors.mediapipe.detector import LandmarksDetector
from pipelines.data.data_module import AVSRDataLoader

detector_labios = LandmarksDetector()
landmarks = detector_labios(VIDEO_PRUEBA)
print(f"Caras detectadas: {sum(1 for x in landmarks if x is not None)}/{len(landmarks)} fotogramas")

cargador_preview = AVSRDataLoader(modality="video", detector="mediapipe", transform=False)
roi = cargador_preview.load_data(VIDEO_PRUEBA, landmarks)
roi = roi.numpy() if hasattr(roi, "numpy") else np.asarray(roi)
print("Forma del recorte (fotogramas, alto, ancho):", roi.shape)

idx = np.linspace(0, len(roi) - 1, 5).astype(int)
cv2.imwrite("/content/roi_labios.png", np.hstack([roi[i] for i in idx]))
print("✅ Guardado /content/roi_labios.png")

In [ ]:
# ✅ VERIFICACIÓN Fase 2 — mira los 5 recortes: ¿se ve la BOCA centrada y nítida?
from PIL import Image
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 3))
plt.imshow(Image.open("/content/roi_labios.png"), cmap="gray")
plt.title("5 recortes de labios (lo que ve el modelo)"); plt.axis("off"); plt.show()
print("Si NO se ve la boca -> vídeo de perfil / mala luz / cara pequeña. Prueba otro.")

## 🗣️ Fase 3 — Inferencia (español + inglés)

> 🧠 **¿Qué es un visema y por qué /p/, /b/ y /m/ se confunden?**
> Un **visema** es "la forma que ponen los labios para un sonido". El problema: /p/, /b/ y /m/ se ven
> **idénticos** (los labios se juntan y se abren igual). Lo que los diferencia está **donde la cámara no
> llega**: /b/ hace vibrar la garganta, /m/ saca aire por la nariz, /p/ ni una cosa ni otra. Es como
> intentar distinguir a tres gemelos que **siempre visten igual**: por fuera no puedes; necesitarías
> oírlos. Por eso leer labios acierta la *forma* pero se equivoca de *letra* muy a menudo.

Para **inglés** usamos el vídeo de demo del repo, cuyo texto real conocemos (sale subtitulado en el GIF).
Para **español** el repo no trae demo: **graba tú un vídeo corto** (frontal, buena luz, 2–4 s) diciendo
una frase, súbelo, y escribe la frase en `TEXTO_REAL_ES`.


In [ ]:
# 3.1 — Descargar inglés y cargar su modelo (el español ya está de la Fase 1).
config_en = preparar_idioma("en")
pipeline_en = InferencePipeline(config_en, detector="mediapipe", face_track=True, device=DEVICE)
print("✅ Modelo INGLÉS cargado.")

In [ ]:
# 3.2 — Acierto aproximado: % de palabras del texto real que aparecen en la transcripción.
def porcentaje_acierto(real, obtenido):
    r = real.lower().split(); o = set(obtenido.lower().split())
    return 100.0 * sum(1 for p in r if p in o) / len(r) if r else 0.0

In [ ]:
# ✅ VERIFICACIÓN Fase 3a — INGLÉS con el vídeo del repo (texto real conocido del GIF).
TEXTO_REAL_EN = ("in any case once this technology really becomes mature and we can truly do "
                 "face and facial expression analysis in the wild")
hyp_en = pipeline_en("/content/test_en.mp4")
print("Real    :", TEXTO_REAL_EN)
print("Obtenido:", hyp_en)
print(f"Acierto aprox: {porcentaje_acierto(TEXTO_REAL_EN, hyp_en):.0f}%  (el repo reporta ~19% de ERROR en su test)")

In [ ]:
# ✅ VERIFICACIÓN Fase 3b — ESPAÑOL con TU vídeo (grábalo: frontal, buena luz, 2-4 s).
VIDEO_PRUEBA_ES = "/content/mi_video_es.mp4"     # <-- sube tu vídeo y ajusta la ruta
TEXTO_REAL_ES   = "escribe aqui la frase que dices en el video"

import os
if os.path.exists(VIDEO_PRUEBA_ES):
    hyp_es = pipeline_es(VIDEO_PRUEBA_ES)
    print("Real    :", TEXTO_REAL_ES)
    print("Obtenido:", hyp_es)
    print(f"Acierto aprox: {porcentaje_acierto(TEXTO_REAL_ES, hyp_es):.0f}%  (el repo reporta ~44.5% de ERROR: fallar casi la mitad es lo esperado)")
else:
    print("⚠ Aún no has subido tu vídeo en español. La fase queda PENDIENTE de este paso")
    print("  (el modelo español ya está cargado y el pipeline verificado con el vídeo en inglés).")

## 🖥️ Fase 4 — Interfaz Gradio (enlace público para el móvil)


In [ ]:
# 4.1 — Instalar Gradio.
%pip install -q gradio
import gradio; print("✅ Gradio", gradio.__version__)

In [ ]:
# 4.2 — La app. Cada modelo se carga la PRIMERA vez que se usa (y queda en caché).
import gradio as gr, numpy as np
from PIL import Image
from pipelines.pipeline import InferencePipeline
from pipelines.detectors.mediapipe.detector import LandmarksDetector
from pipelines.data.data_module import AVSRDataLoader

_pipes = {"es": pipeline_es, "en": pipeline_en}   # ya cargados en Fases 1 y 3
_detector_labios = LandmarksDetector()
_cargador_preview = AVSRDataLoader(modality="video", detector="mediapipe", transform=False)

# Fase 4: SOLO los dos idiomas ya verificados (la Fase 5 amplía la lista).
ACTIVOS = {"es": "Español", "en": "Inglés"}

def _get_pipe(cod):
    if cod not in _pipes:
        cfg = preparar_idioma(cod)          # descarga bajo demanda
        _pipes[cod] = InferencePipeline(cfg, detector="mediapipe", face_track=True, device=DEVICE)
    return _pipes[cod]

def _preview_labios(ruta):
    landmarks = _detector_labios(ruta)
    roi = _cargador_preview.load_data(ruta, landmarks)
    roi = np.asarray(roi.numpy() if hasattr(roi, "numpy") else roi)
    idx = np.linspace(0, len(roi) - 1, 5).astype(int)
    return Image.fromarray(np.hstack([roi[i] for i in idx]))

def transcribir(video, idioma_nombre):
    if not video:
        return None, "Sube un vídeo primero."
    cod = [k for k, v in ACTIVOS.items() if v == idioma_nombre][0]
    try:
        preview = _preview_labios(video)
    except Exception as e:
        return None, f"No pude detectar la boca (¿perfil / poca luz?): {e}"
    return preview, _get_pipe(cod)(video)

with gr.Blocks(title="Lectura de labios (VSR)") as demo:
    gr.Markdown("# 👄 Lectura de labios multilingüe\nSube un vídeo **frontal y con buena luz**. "
                "Transcribe SIN audio. Es una demo: se equivoca mucho, sobre todo en español.")
    with gr.Row():
        with gr.Column():
            v = gr.Video(label="Vídeo (.mp4 / .avi)")
            idi = gr.Dropdown(choices=list(ACTIVOS.values()), value="Español", label="Idioma")
            btn = gr.Button("Transcribir", variant="primary")
        with gr.Column():
            img = gr.Image(label="Recorte de labios que ve el modelo")
            out = gr.Textbox(label="Transcripción", lines=4)
    btn.click(transcribir, [v, idi], [img, out])

demo.launch(share=True)   # <-- enlace https://....gradio.live para el móvil

**✅ VERIFICACIÓN Fase 4:** abre el enlace `gradio.live` en el móvil y prueba: el vídeo
`/content/test_en.mp4` (descárgalo del panel de archivos o usa uno tuyo) en inglés, y tu vídeo en español.
Debes ver el recorte de labios y una transcripción.


## 🌍 Fase 5 — Resto de idiomas (bajo demanda)

Añadimos **francés, portugués y mandarín**. Los pesos se descargan **la primera vez** que eliges el
idioma. El **italiano no existe** en este repo (sin pesos públicos): no está en el selector.

Para el **francés** tienes vídeo de prueba con texto conocido: `/content/test_fr.mp4` (del GIF de demo
del repo). Su subtítulo dice: *"et ensuite on va créer une administration il va y avoir une
administration qui s'en occupe"*. Para portugués y mandarín necesitarás vídeos tuyos o de internet.


In [ ]:
# 5.1 — Ampliar el selector y relanzar la app (transcribir() ya descarga bajo demanda).
ACTIVOS = {"es": "Español", "en": "Inglés", "fr": "Francés", "pt": "Portugués", "zh": "Mandarín"}
demo.close()
demo.launch(share=True)

In [ ]:
# 5.2 — (Opcional, más rápido que la app) Verificar el FRANCÉS aquí mismo con su vídeo del repo.
TEXTO_REAL_FR = "et ensuite on va créer une administration il va y avoir une administration qui sen occupe"
pipe_fr = _get_pipe("fr")
hyp_fr = pipe_fr("/content/test_fr.mp4")
print("Real    :", TEXTO_REAL_FR)
print("Obtenido:", hyp_fr)
print(f"Acierto aprox: {porcentaje_acierto(TEXTO_REAL_FR, hyp_fr):.0f}%  (el repo reporta ~58.6% de ERROR en francés)")

**✅ VERIFICACIÓN Fase 5:** al menos un vídeo probado por idioma — francés ya tiene el del repo;
portugués y mandarín, con vídeos tuyos (frontal, buena luz).


## 📊 Calidad esperada por idioma (datos del propio repo) y qué NO funciona

**Tasa de ERROR de palabra (WER) que reporta el repo** — cuanto más bajo, mejor:

| Idioma    | Config usada                    | Error aprox. | Lectura honesta |
|-----------|---------------------------------|:-----------:|-----------------|
| Inglés    | `LRS3_V_WER19.1.ini`            | ~19–20%     | El mejor; aun así 1 de cada 5 palabras mal |
| Español   | `CMUMOSEAS_V_ES_WER44.5.ini`    | ~44.5%      | **Falla casi la mitad de las palabras** |
| Portugués | `CMUMOSEAS_V_PT_WER51.4.ini`    | ~51%        | Falla más de la mitad |
| Francés   | `CMUMOSEAS_V_FR_WER58.6.ini`    | ~59%        | Falla la mayoría |
| Mandarín  | `CMLR_V_WER8.0.ini`             | ~8% (por carácter) | Bajo por medirse por carácter, no por palabra |
| Italiano  | — no disponible —               | —           | **Sin pesos públicos en este repo** |

**Esto es una demo del estado del arte público, NO un transcriptor fiable.**

**Qué NO funciona:**
- **Caras de perfil** o giradas; **mala luz** o boca en sombra.
- **Varios hablantes** en el cuadro; boca **tapada** (mano, micro, mascarilla).
- Sonidos que se ven igual (**/p/ /b/ /m/**): se confunden por diseño (ver Fase 3).
- Los WER del repo son sobre SUS vídeos de test; con vídeos caseros el error suele ser mayor.

### Ideas si quieres mejor resultado
- Frontal, buena luz, vocalizando, frases cortas. Inglés rinde mucho mejor que español.
- Un modelo **audiovisual** (con audio) acierta muchísimo más, pero ya no es "leer labios".
